# Session 2: open the network up

**What you will see.** We take three snapshots of the network from session 1: before training (epoch 0), on the flat stretch (epoch 30), and at the end (epoch 400). We test them three ways, from the outside in: ask them questions, change their inputs, and read their connection strengths.

The goal is to answer the question from the break: what did the network know at epoch 30, and what did it learn after that?

In [ ]:
# Setup: find the course files, downloading them first if this is Colab.
import os, sys, shutil
REPO = "https://github.com/icarm/SACNAS-2026"
if "google.colab" in sys.modules and not os.path.exists("/content/course/mlmath.py"):
    shutil.rmtree("/content/course", ignore_errors=True)
    if os.system(f"git clone -q --depth 1 {REPO} /content/course") != 0:
        raise RuntimeError("Could not download the course files. Check the wifi, then run this cell again.")
for folder in ["/content/course", ".", ".."]:
    if os.path.exists(os.path.join(folder, "mlmath.py")):
        os.chdir(folder)
        break
else:
    raise RuntimeError("Could not find the course files. Open this notebook from the ml4math-mini-course folder.")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Ready.")

This cell loads the three snapshots. They come from the same training run as session 1, saved in advance so everyone starts from the same place.

In [ ]:
import matplotlib.pyplot as plt
import mlmath

data = mlmath.make_data(modulus=6)
models = {f"epoch {e}": mlmath.load_model(f"checkpoints/mod6_epoch{e:03d}.pt") for e in (0, 30, 400)}
history = mlmath.load_history("checkpoints/mod6_history.csv")

mlmath.plot_history(history, marks=(30,))
plt.show()

The vertical line marks epoch 30, the middle snapshot.

## Probe 1: ask it

The simplest test of all. Give each snapshot the same number and look at how confident it is in each answer. The green bar is the correct answer.

**Look for:** at epoch 30, which answers does the network consider? Which does it rule out completely?

In [ ]:
mlmath.ask(models, 4712)

Try your own numbers. Change the list and run the cell again. Try an odd number. Try your birth year.

In [ ]:
for n in [2026, 9999, 3]:
    mlmath.ask(models, n)

Three numbers are anecdotes. This cell asks about all 5,000 test numbers and tallies the answers. Each row is a true answer, each column is what the network said. A perfect network has everything on the diagonal.

**Look for:** in the epoch 30 table, which cells are empty? What do the empty cells have in common?

In [ ]:
mlmath.plot_confusions(models, data)

**What this shows.** At epoch 30 the network never calls an even number odd or an odd number even. It has learned parity perfectly and nothing else. That is the first drop in the loss, and it explains the flat stretch at ln 3: for each number, three answers remain.

## Probe 2: change one digit

Now an experiment with a control. Take every number, change one digit, and check whether the network changes its answer.

Adding 3 to the thousands digit adds 3000 to n. Since 3000 is divisible by 6, the right answer does not change. Adding 1 adds 1000, and 1000 mod 6 = 4, so the right answer must change.

**Predict first.** For a network that really computes n mod 6, what should the two percentages below be?

In [ ]:
print("How often the answer stays the same when we change the thousands digit")
print("            add 3 (should stay)   add 1 (should change)")
for name, model in models.items():
    stay = mlmath.swap_test(model, data, position=0, shift=3)
    move = mlmath.swap_test(model, data, position=0, shift=1)
    print(f"{name:10s}  {stay:14.0%}  {move:20.0%}")

**Discuss.** At epoch 30 the two numbers are nearly equal. What does that tell you about how the network uses the thousands digit at that stage?

The next cell runs the same experiment for every pair of digits. Row a, column b: replace thousands digit a with b, and record how often (in percent) the answer survives.

**Look for:** in the epoch 400 table, which digits can be swapped freely? Write down the groups.

In [ ]:
after = {name: models[name] for name in ("epoch 30", "epoch 400")}
mlmath.plot_swap_tables(after, data, position=0)

**What this shows.** By epoch 400 the network treats 0, 3, 6, 9 as the same digit in the thousands place. Likewise 1, 4, 7 and 2, 5, 8. Nobody told it about remainders mod 3. It sorted the digits into those classes because that is what the task needs. This is the second drop in the loss.

Now the same table for the ones digit. This digit is special, since it also decides parity.

**Predict first.** Which ones digits should be interchangeable for n mod 6?

In [ ]:
mlmath.plot_swap_tables(after, data, position=3)

**Discuss.** At epoch 400 you should see the pairs 0 and 6, 1 and 7, 2 and 8, 3 and 9, with 4 and 5 on their own. Why those pairs? And why does the epoch 30 table look like a checkerboard?

## Probe 3: read the connection strengths

Probes 1 and 2 treat the network as a sealed box. Now we look inside. Each digit in each place sends a signal to the 512 hidden units, and each such signal has a strength. If two digits send nearly the same pattern of strengths, the network cannot tell them apart.

The tables below compare those patterns for each pair of digits. Blue means similar, orange means opposite. The right column is the ones digit. The left column is the other three digits, averaged.

**Look for:** the checkerboard at epoch 30. Then, at epoch 400, the same groups you found in Probe 2.

In [ ]:
mlmath.plot_similarities(after, positions=((0, 1, 2), 3))

**An honest note.** The groups {0, 3, 6, 9}, {1, 4, 7}, {2, 5, 8} are visible in the bottom-left table, but faintly, and much less sharply than in the swap test. That is typical. What a network does is often easier to pin down than how its insides are arranged, and the two kinds of evidence are worth keeping apart.

## Your turn

The final network is right about 99% of the time on test numbers, not 100%. This cell lists the test numbers it gets wrong.

**Look for:** is there any pattern in its mistakes? Pick one and ask the network about it with `mlmath.ask(models, n)`.

In [ ]:
wrong = mlmath.wrong_numbers(models["epoch 400"], data)
print(len(wrong), "test numbers wrong out of", len(data.test))
print(wrong)

## What we did

1. Built a dataset where we could check every label ourselves.
2. Tried simple encodings first and watched them fail, which told us why the encoding matters.
3. Trained a network and saw the loss fall in two steps.
4. Tested what the network knew at each stage: first parity, then remainders mod 3. Those are the two pieces of the Chinese Remainder Theorem, learned in order of difficulty.

One caution to take home. The network is 99% right on numbers drawn the same way as its training numbers. That is evidence about those numbers. It is not a proof, and experiment D in the explore notebook shows how easily it breaks.

If there is time, open **03_explore.ipynb**.